# Phase 6 · S1 generation tuning — same trained model, better decoding (no training)

S1 (A + LORMA) scored BERTScore F1 0.8287 / ROUGE-1 0.473 / ROUGE-L 0.235 on Codabench. Its validation output showed two decoding problems:
- **summaries too short:** ROUGE rose with every extra sentence kept, and at "keep all" the summaries were still only 0.88× the reference length (minimum length per chunk was 8% of the chunk);
- **repetition:** 20% of word trigrams repeated (e.g. "the Tribunal held that the Tribunal …").

This notebook **does not train anything**. It loads S1's trained LORMA weights and prepared chunks from `Phase6_S1_Dataset` and changes only the decoding:
1. **Minimum summary length per chunk:** 8% (S1's setting), 15% or 22% of the chunk's tokens (maximum raised to 50%).
2. **Repetition penalty** on tokens already generated for that chunk (Keskar et al., 2019): 1.0 (none, S1's setting) or 1.2; 4-gram blocking kept.

The 6 combinations are compared on 60 of S1's held-out validation documents (ROUGE-1 + ROUGE-L); the best is re-scored on all 150 validation documents (directly comparable with S1's 0.461 / 0.245) and used for the 100 test summaries. Changing only generation settings is not a new experiment — it is an improved S1 submission.

**Before running (Kaggle)**
- Accelerator: **GPU T4** (one is enough) · internet not needed.
- Add Input → **`Phase6_S1_Dataset`** (S1's output: `P6_S1_A_LORMA/checkpoints/best.pt` and `data_cache.pt`), **`rohitbatra2004/phase111`** (Architecture A) and **`rohitbatra2004/cs781-phase1-tokenized-corpus`** (tokenizer).
- Expected time ≈ 60–80 minutes.

Outputs: `/kaggle/working/P6_S1_tuned/` — upload `submission.zip`.

In [ ]:
# Cell 1 — Imports, random seed and device.
# Every later cell uses the names defined here.
import os, json, glob, math, time, random, shutil, zipfile
from pathlib import Path
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import hashlib, re
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report, confusion_matrix
from tokenizers import Tokenizer
from huggingface_hub import snapshot_download

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch", torch.__version__, "| device:", DEVICE, "| GPUs visible:", torch.cuda.device_count())

In [ ]:
# Cell 2 — Paths and settings. Model / data settings must equal S1's (checked against S1's saved data); only decoding changes.
RUN_NAME = "P6_S1_tuned"
SOURCE_RUN = "P6_S1_A_LORMA"
WORK_DIR = Path("/kaggle/working") / RUN_NAME
OUT_DIR = WORK_DIR / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# S1's settings (needed to rebuild the model and read its data).
CHUNK_TOKENS = 1200
MAX_TARGET_TOKENS = 640
SEPARATOR = " Summary:"
LORMA_RANK = 8
LORMA_ALPHA = 16.0
LORMA_TARGETS_ATTN = ("q_proj", "k_proj", "v_proj", "out_proj")
LORMA_TARGETS_FFN = ("fc1", "fc2")

# Decoding (the only thing tuned here).
GEN_BATCH = 16
NO_REPEAT_NGRAM = 4
MAX_NEW_FRACTION = 0.50                     # maximum summary tokens per chunk = 50% of the chunk (+ 32), ≤ MAX_TARGET_TOKENS
MIN_NEW_FRACTIONS = (0.08, 0.15, 0.22)      # minimum summary tokens per chunk (S1 used 0.08)
REPETITION_PENALTIES = (1.0, 1.2)           # penalty on tokens already generated for the chunk (S1 used none = 1.0)
TUNE_DOCS = 60                              # validation documents used to compare the settings
USE_AMP = DEVICE.type == "cuda"
print("Run:", RUN_NAME, "| outputs:", WORK_DIR)

In [ ]:
# Cell 3 — Load S1's prepared data (chunks, held-out validation documents with references, test chunks) and its best weights.
src = sorted(glob.glob(f"/kaggle/input/**/{SOURCE_RUN}/data_cache.pt", recursive=True))
ckpt = sorted(glob.glob(f"/kaggle/input/**/{SOURCE_RUN}/checkpoints/best.pt", recursive=True))
assert src and ckpt, "Add S1's output dataset (Phase6_S1_Dataset) as an input."
data = torch.load(src[0], weights_only=False)
assert data["settings"] == {"chunk_tokens": CHUNK_TOKENS, "max_target_tokens": MAX_TARGET_TOKENS, "separator": SEPARATOR}
best = torch.load(ckpt[0], map_location="cpu", weights_only=False)
print(f"S1 data: {len(data['val'])} validation documents, {len(data['test'])} test documents | S1 best epoch {best['epoch']} "
      f"(validation loss {best['val_loss']:.4f})")

In [ ]:
# Cell 4 — Tokenizer, separator and the sentence / word patterns used by S1 (for decoding and scoring).
tokenizer = Tokenizer.from_file(sorted(glob.glob("/kaggle/input/**/tokenizer/tokenizer.json", recursive=True))[0])
PAD_ID, BOS_ID, EOS_ID, UNK_ID = (tokenizer.token_to_id(t) for t in ("[PAD]", "[BOS]", "[EOS]", "[UNK]"))
assert (PAD_ID, UNK_ID, BOS_ID, EOS_ID) == (0, 1, 2, 3)
SEP_IDS = tokenizer.encode(SEPARATOR).ids
SENT_SPLIT = re.compile(r"(?<=[.?!])\s+(?=[A-Z0-9(\"'])")
WORD = re.compile(r"[a-z0-9]+")

def split_sentences(text):
    return [s for s in SENT_SPLIT.split(text) if s.strip()]
print("Tokenizer ready | separator ids", SEP_IDS)

In [ ]:
# Cell 5 — Phase-1 Architecture A classes and the pretrained weights, verified against the Phase-1 metadata.
# The weights stay in memory so that build_model() can create a fresh adapted copy.
a_paths = sorted(glob.glob("/kaggle/input/**/architecture_A/model_A_state_dict.pt", recursive=True))
assert a_paths, "Add the dataset rohitbatra2004/phase111 as an input."
A_ROOT = Path(a_paths[0]).parent
A_METADATA = json.load(open(A_ROOT / "model_A_metadata.json"))
A_CONFIG = A_METADATA["model_config"]
assert A_CONFIG["pre_norm"] and A_CONFIG["tie_embeddings"], "Phase-1 A is a pre-LN decoder with tied embeddings"

class CausalSelfAttention(nn.Module):
    """Same parameters as Phase 1; uses PyTorch's fused causal attention for speed and memory."""
    def __init__(self, cfg):
        super().__init__()
        d = cfg["d_model"]
        self.n_heads, self.dropout = cfg["n_heads"], cfg["dropout"]
        self.q_proj = nn.Linear(d, d, bias=cfg["use_bias"])
        self.k_proj = nn.Linear(d, d, bias=cfg["use_bias"])
        self.v_proj = nn.Linear(d, d, bias=cfg["use_bias"])
        self.out_proj = nn.Linear(d, d, bias=cfg["use_bias"])
        self.resid_dropout = nn.Dropout(cfg["dropout"])

    def forward(self, x):
        B, T, C = x.shape
        split = lambda t: t.view(B, T, self.n_heads, C // self.n_heads).transpose(1, 2)
        out = F.scaled_dot_product_attention(split(self.q_proj(x)), split(self.k_proj(x)), split(self.v_proj(x)),
                                             dropout_p=self.dropout if self.training else 0.0, is_causal=True)
        return self.resid_dropout(self.out_proj(out.transpose(1, 2).reshape(B, T, C)))

class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.fc1 = nn.Linear(cfg["d_model"], cfg["d_ff"], bias=cfg["use_bias"])
        self.fc2 = nn.Linear(cfg["d_ff"], cfg["d_model"], bias=cfg["use_bias"])
        self.dropout = nn.Dropout(cfg["dropout"])

    def forward(self, x):
        return self.dropout(self.fc2(F.gelu(self.fc1(x))))

class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln_1 = nn.LayerNorm(cfg["d_model"])
        self.attention = CausalSelfAttention(cfg)
        self.ln_2 = nn.LayerNorm(cfg["d_model"])
        self.feed_forward = FeedForward(cfg)

    def forward(self, x):
        x = x + self.attention(self.ln_1(x))
        return x + self.feed_forward(self.ln_2(x))

class ArchitectureABackbone(nn.Module):
    """Phase-1 decoder-only LM without the vocabulary projection.
    Returns the final-normalised state of every token: [batch, tokens, d_model]."""
    def __init__(self, cfg):
        super().__init__()
        self.token_embedding = nn.Embedding(cfg["vocab_size"], cfg["d_model"])
        self.position_embedding = nn.Embedding(cfg["context_length"], cfg["d_model"])
        self.embedding_dropout = nn.Dropout(cfg["dropout"])
        self.blocks = nn.ModuleList([TransformerBlock(cfg) for _ in range(cfg["n_layers"])])
        self.final_norm = nn.LayerNorm(cfg["d_model"])
        # Tied output weight, kept so that the saved Phase-1 state dict loads strictly; unused here.
        self.output_weight = self.token_embedding.weight

    def forward(self, input_ids):
        positions = torch.arange(input_ids.size(1), device=input_ids.device).unsqueeze(0)
        x = self.embedding_dropout(self.token_embedding(input_ids) + self.position_embedding(positions))
        for block in self.blocks:
            x = block(x)
        return self.final_norm(x)

saved = torch.load(a_paths[0], map_location="cpu", weights_only=True)
# The export is either the raw state dict or a dict that wraps it under "state_dict".
PHASE1_STATE = saved["state_dict"] if "state_dict" in saved else saved
del saved

check = ArchitectureABackbone(A_CONFIG)
check.load_state_dict(PHASE1_STATE, strict=True)
n_backbone = sum(p.numel() for p in check.parameters())
assert n_backbone == int(A_METADATA["parameter_count"]), "Parameter count differs from Phase-1 metadata"
del check
print(json.dumps(A_CONFIG, indent=2))
print(f"Phase-1 Architecture A weights verified from {A_ROOT} — {n_backbone:,} parameters")

In [ ]:
# Cell 6 — LORMA: multiplicative low-rank update of a frozen linear layer, W_eff = (I + (α/r)·B·A)·W0.
# Applied to q/k/v/out and fc1/fc2 in all 12 layers; B = 0 at start, so the model initially equals Phase 1.
class LoRMALinear(nn.Module):
    def __init__(self, base, rank, alpha):
        super().__init__()
        self.base = base
        self.lorma_A = nn.Parameter(torch.empty(rank, base.out_features))
        self.lorma_B = nn.Parameter(torch.zeros(base.out_features, rank))
        self.scale = alpha / rank
        nn.init.kaiming_uniform_(self.lorma_A, a=math.sqrt(5))

    def forward(self, x):
        w0x = F.linear(x, self.base.weight)                               # W0 x (without bias)
        update = F.linear(F.linear(w0x, self.lorma_A), self.lorma_B)       # B (A (W0 x))
        out = w0x + self.scale * update
        return out + self.base.bias if self.base.bias is not None else out

def make_lorma_backbone():
    bb = ArchitectureABackbone(A_CONFIG)
    bb.load_state_dict(PHASE1_STATE, strict=True)
    for p in bb.parameters():
        p.requires_grad_(False)
    for block in bb.blocks:
        for name in LORMA_TARGETS_ATTN:
            setattr(block.attention, name, LoRMALinear(getattr(block.attention, name), LORMA_RANK, LORMA_ALPHA))
        for name in LORMA_TARGETS_FFN:
            setattr(block.feed_forward, name, LoRMALinear(getattr(block.feed_forward, name), LORMA_RANK, LORMA_ALPHA))
    return bb

# Checks on the device: (1) with B = 0 the output equals the frozen Phase-1 model; (2) A and B receive gradients.
probe_ids = torch.randint(4, 1000, (2, 16), device=DEVICE)
plain = ArchitectureABackbone(A_CONFIG); plain.load_state_dict(PHASE1_STATE, strict=True); plain = plain.to(DEVICE).eval()
probe_bb = make_lorma_backbone().to(DEVICE).eval()
with torch.no_grad():
    assert torch.allclose(plain(probe_ids), probe_bb(probe_ids), atol=1e-4), "LORMA with B = 0 must equal Phase 1"
    lorma_layers = [m for m in probe_bb.modules() if isinstance(m, LoRMALinear)]
    for m in lorma_layers:
        m.lorma_B.normal_(0, 1e-3)                                       # temporary non-zero B for the gradient check
probe_bb.train()
probe_bb(probe_ids).float().pow(2).mean().backward()
assert all(m.lorma_A.grad is not None and m.lorma_A.grad.abs().sum() > 0 for m in lorma_layers), "LORMA gets no gradient"
n_lorma = sum(p.numel() for p in probe_bb.parameters() if p.requires_grad)
print(f"LORMA on {len(lorma_layers)} layers — {n_lorma:,} trainable backbone parameters ({100 * n_lorma / n_backbone:.2f}%)")
del plain, probe_bb, lorma_layers
torch.cuda.empty_cache()

In [ ]:
# Cell 7 — Rebuild S1's model (Phase-1 A + LORMA) and load S1's trained LORMA weights (strict check of the names).
class SummaryLM(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone

model = SummaryLM(make_lorma_backbone()).to(DEVICE)
expected = {n for n, p in model.named_parameters() if p.requires_grad}
assert set(best["trainable"]) == expected, "Checkpoint does not match S1's LORMA model"
model.load_state_dict(best["trainable"], strict=False)
model.eval()
CONTEXT = A_CONFIG["context_length"]
print(f"S1 model ready: {sum(p.numel() for p in model.parameters()):,} parameters, {len(expected)} LORMA tensors loaded")

In [ ]:
# Cell 8 — S1's cached (KV) batched greedy generation, plus an optional repetition penalty; min / max lengths per chunk.
# Prompts are left-padded; padding keys are masked and positions start at each prompt's first real token.
def attend(attn, h, key_mask, cache):
    B, T, Cd = h.shape
    split = lambda t: t.view(B, T, attn.n_heads, Cd // attn.n_heads).transpose(1, 2)
    q, k, v = split(attn.q_proj(h)), split(attn.k_proj(h)), split(attn.v_proj(h))
    if cache is not None:
        k, v = torch.cat([cache[0], k], 2), torch.cat([cache[1], v], 2)
    S = k.size(2)
    mask = key_mask[:, None, None, :S].expand(B, 1, T, S)
    if T > 1:                                                     # prefill: causal + padding mask (a query always sees itself)
        causal = torch.ones(T, S, dtype=torch.bool, device=h.device).tril(S - T)
        mask = (mask & causal) | torch.eye(T, S, dtype=torch.bool, device=h.device)
    out = F.scaled_dot_product_attention(q, k, v, attn_mask=mask)
    return attn.out_proj(out.transpose(1, 2).reshape(B, T, Cd)), (k, v)

def run_layers(bb, x, key_mask, caches):
    new = []
    for block, cache in zip(bb.blocks, caches):
        a, kv = attend(block.attention, block.ln_1(x), key_mask, cache)
        x = x + a
        x = x + block.feed_forward(block.ln_2(x))
        new.append(kv)
    return bb.final_norm(x), new

def block_repeats(logits, generated, n):
    """Forbid tokens that would repeat an n-gram already generated in that row."""
    for b, seq in enumerate(generated):
        if len(seq) >= n - 1:
            prefix = tuple(seq[-(n - 1):])
            banned = [seq[i + n - 1] for i in range(len(seq) - n + 1) if tuple(seq[i:i + n - 1]) == prefix]
            if banned:
                logits[b, banned] = -1e4

@torch.no_grad()
def generate(bb, prompts, max_new, min_new, repetition_penalty=1.0):
    """prompts: token lists; max_new / min_new: per-prompt limits → generated token lists (without [EOS])."""
    bb.eval()
    B, L = len(prompts), max(len(p) for p in prompts)
    ids = torch.full((B, L), PAD_ID, dtype=torch.long, device=DEVICE)
    for b, p in enumerate(prompts):
        ids[b, L - len(p):] = torch.tensor(p, device=DEVICE)
    key_mask = ids.ne(PAD_ID)
    positions = (key_mask.long().cumsum(1) - 1).clamp(min=0)
    weight = bb.token_embedding.weight
    with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
        x = bb.token_embedding(ids) + bb.position_embedding(positions)
        states, caches = run_layers(bb, x, key_mask, [None] * len(bb.blocks))
        next_pos = positions[:, -1] + 1
        generated, done = [[] for _ in range(B)], [False] * B
        logits = F.linear(states[:, -1], weight).float()
        for step in range(max(max_new)):
            logits[:, [PAD_ID, BOS_ID, UNK_ID]] = -1e4
            for b in range(B):
                if step < min_new[b]:
                    logits[b, EOS_ID] = -1e4
            if repetition_penalty != 1.0:                         # CTRL-style penalty on tokens already generated
                for b, seq in enumerate(generated):
                    if seq:
                        idx = torch.tensor(sorted(set(seq)), device=DEVICE)
                        v = logits[b, idx]
                        logits[b, idx] = torch.where(v > 0, v / repetition_penalty, v * repetition_penalty)
            block_repeats(logits, generated, NO_REPEAT_NGRAM)
            token = logits.argmax(-1)
            for b in range(B):
                if not done[b]:
                    if token[b].item() == EOS_ID or len(generated[b]) >= max_new[b]:
                        done[b] = True
                    else:
                        generated[b].append(token[b].item())
            if all(done):
                break
            key_mask = torch.cat([key_mask, torch.ones(B, 1, dtype=torch.bool, device=DEVICE)], 1)
            x = bb.token_embedding(token[:, None]) + bb.position_embedding(next_pos.clamp(max=CONTEXT - 1)[:, None])
            states, caches = run_layers(bb, x, key_mask, caches)
            next_pos = next_pos + 1
            logits = F.linear(states[:, -1], weight).float()
    return generated

def summarise(items, min_fraction, repetition_penalty, first_chunk_only=False):
    """Per document: list of per-chunk summaries (each a list of sentences)."""
    jobs = [(i, j, c) for i, d in enumerate(items) for j, c in enumerate(d["chunks"][:1] if first_chunk_only else d["chunks"])]
    jobs.sort(key=lambda job: len(job[2]))
    out = [dict() for _ in items]
    for s in range(0, len(jobs), GEN_BATCH):
        batch = jobs[s:s + GEN_BATCH]
        prompts = [[BOS_ID] + c.tolist() + SEP_IDS for _, _, c in batch]
        max_new = [min(MAX_TARGET_TOKENS, int(MAX_NEW_FRACTION * len(c)) + 32) for _, _, c in batch]
        min_new = [int(min_fraction * len(c)) for _, _, c in batch]
        for (i, j, _), g in zip(batch, generate(model.backbone, prompts, max_new, min_new, repetition_penalty)):
            out[i][j] = split_sentences(tokenizer.decode(g).strip())
    return [[out[i][j] for j in sorted(out[i])] for i in range(len(items))]

def join_summary(chunk_summaries, keep_fraction=1.0):
    """Keep the first share of each chunk summary's sentences, drop sentences already used, join in document order."""
    seen, kept = set(), []
    for sentences in chunk_summaries:
        for s in sentences[:max(1, math.ceil(keep_fraction * len(sentences)))] if sentences else []:
            key = " ".join(WORD.findall(s.lower()))
            if key and key not in seen:
                seen.add(key); kept.append(s)
    return " ".join(kept)
print("Generation helpers ready.")

In [ ]:
# Cell 9 — ROUGE-1 / ROUGE-L F1 (word level, lower-cased) and repetition, for the held-out validation documents.
def rouge1_f(ref, hyp):
    overlap = sum((Counter(ref) & Counter(hyp)).values())
    return 0.0 if overlap == 0 else 2 * overlap / (len(ref) + len(hyp))

def rougeL_f(ref, hyp, cap=800):
    ref, hyp = ref[:cap], hyp[:cap]
    prev = [0] * (len(hyp) + 1)
    for x in ref:
        cur = [0]
        for j, y in enumerate(hyp, 1):
            cur.append(prev[j - 1] + 1 if x == y else max(prev[j], cur[j - 1]))
        prev = cur
    return 0.0 if prev[-1] == 0 else 2 * prev[-1] / (len(ref) + len(hyp))

def score_summaries(refs, hyps):
    r1, rl, lengths, rep = [], [], [], []
    for ref, hyp in zip(refs, hyps):
        rw, hw = WORD.findall(ref.lower()), WORD.findall(hyp.lower())
        r1.append(rouge1_f(rw, hw)); rl.append(rougeL_f(rw, hw)); lengths.append(len(hw) / max(len(rw), 1))
        grams = list(zip(hw, hw[1:], hw[2:]))
        rep.append(1 - len(set(grams)) / max(len(grams), 1))
    return {"rouge1": float(np.mean(r1)), "rougeL": float(np.mean(rl)), "length_ratio": float(np.median(lengths)),
            "repeated_trigram_share": float(np.mean(rep))}

def describe(s):
    return (f"ROUGE-1 {s['rouge1']:.4f} | ROUGE-L {s['rougeL']:.4f} | length vs reference ×{s['length_ratio']:.2f} | "
            f"repeated trigrams {100 * s['repeated_trigram_share']:.1f}%")
print("Scoring ready (validation only).")

In [ ]:
# Cell 10 — Compare the decoding settings on TUNE_DOCS validation documents (keep all sentences of every chunk summary).
tune_items = data["val"][:TUNE_DOCS]
tune_refs = [d["summary"] for d in tune_items]
grid = {}
for min_fraction in MIN_NEW_FRACTIONS:
    for penalty in REPETITION_PENALTIES:
        start_time = time.time()
        summaries = [join_summary(c) for c in summarise(tune_items, min_fraction, penalty)]
        grid[(min_fraction, penalty)] = score_summaries(tune_refs, summaries)
        print(f"  min length {min_fraction:.2f} · repetition penalty {penalty:.1f}: {describe(grid[(min_fraction, penalty)])} "
              f"| {(time.time() - start_time) / 60:.1f} min", flush=True)
BEST_MIN, BEST_PENALTY = max(grid, key=lambda k: grid[k]["rouge1"] + grid[k]["rougeL"])
S1_SETTING = (MIN_NEW_FRACTIONS[0], REPETITION_PENALTIES[0])          # 0.08 / 1.0 = what S1 used
print(f"Chosen: minimum length {BEST_MIN:.2f} of each chunk, repetition penalty {BEST_PENALTY:.1f} "
      f"(S1's setting {S1_SETTING[0]:.2f} / {S1_SETTING[1]:.1f} scored {grid[S1_SETTING]['rouge1']:.4f} / "
      f"{grid[S1_SETTING]['rougeL']:.4f} on the same documents)")

In [ ]:
# Cell 11 — Re-score the chosen setting on all held-out validation documents (comparable with S1's 0.4612 / 0.2452).
refs = [d["summary"] for d in data["val"]]
start_time = time.time()
val_chunk_summaries = summarise(data["val"], BEST_MIN, BEST_PENALTY)
val_result = score_summaries(refs, [join_summary(c) for c in val_chunk_summaries])
print(f"All {len(refs)} validation documents: {describe(val_result)} | {(time.time() - start_time) / 60:.1f} min")
print("\nExample (validation) — reference:\n", data["val"][0]["summary"][:700],
      "\n\nGenerated:\n", join_summary(val_chunk_summaries[0])[:700])

In [ ]:
# Cell 12 — Test summaries with the chosen setting, format checks, submission.zip and a run summary.
start_time = time.time()
test_chunk_summaries = summarise(data["test"], BEST_MIN, BEST_PENALTY)
prediction = [{"id": d["id"], "summary": join_summary(c)} for d, c in zip(data["test"], test_chunk_summaries)]
for p, d in zip(prediction, data["test"]):
    if not p["summary"].strip():
        p["summary"] = tokenizer.decode(d["chunks"][0][:200].tolist())
assert len({p["id"] for p in prediction}) == len(prediction) == len(data["test"])
assert all(isinstance(p["summary"], str) and p["summary"].strip() for p in prediction)
json.dump(prediction, open(WORK_DIR / "prediction.json", "w"), indent=1)
with zipfile.ZipFile(WORK_DIR / "submission.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(WORK_DIR / "prediction.json", arcname="prediction.json")
with zipfile.ZipFile(WORK_DIR / "submission.zip") as zf:
    assert zf.namelist() == ["prediction.json"]
words = np.array([len(p["summary"].split()) for p in prediction])
json.dump({"source": SOURCE_RUN, "min_new_fraction": BEST_MIN, "repetition_penalty": BEST_PENALTY, "max_new_fraction": MAX_NEW_FRACTION,
           "grid": {f"{k[0]}_{k[1]}": v for k, v in grid.items()}, "validation_all": val_result,
           "test_words_median": int(np.median(words))}, open(OUT_DIR / "run_summary.json", "w"), indent=2)
print(f"submission.zip: {len(prediction)} summaries | words per summary median {int(np.median(words))} (S1: 581; references ≈ 640) "
      f"| generation {(time.time() - start_time) / 60:.1f} min")